# 🔊 BookVoice — Stimme testen (aus Drive-Checkpoint)
**v1.0 · Juni 2026**

Lädt einen fine-getunten XTTS-Checkpoint **direkt aus deinem Google Drive** und spricht Test-Text in deiner Stimme. Kein Upload des Modells nötig.

➡️ Runtime → **T4 GPU**. Zellen 1→3 der Reihe nach.


In [ ]:
#@title 1. Installieren { display-mode: "form" }
print("⏳ coqui-tts ...")
!pip install -q coqui-tts "transformers>=4.45,<4.57"
!pip uninstall -q -y torchvision
import torch
print("✅ CUDA:", torch.cuda.is_available(), "·", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
#@title 2. Checkpoint aus Drive + Basis-Dateien { display-mode: "form" }
import os
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CHECKPOINT_PFAD = "/content/drive/MyDrive/checkpoint_2000.pth"  #@param {type:"string"}
assert os.path.exists(CHECKPOINT_PFAD), f"❌ Nicht gefunden: {CHECKPOINT_PFAD} — Pfad in Drive prüfen!"

from huggingface_hub import hf_hub_download
CKPT_DIR = "/content/xtts_base"; os.makedirs(CKPT_DIR, exist_ok=True)
print("⏳ Lade Basis-config + vocab von HuggingFace ...")
CONFIG = hf_hub_download("coqui/XTTS-v2", "config.json", local_dir=CKPT_DIR)
VOCAB  = hf_hub_download("coqui/XTTS-v2", "vocab.json",  local_dir=CKPT_DIR)
print(f"✅ Checkpoint: {CHECKPOINT_PFAD} ({os.path.getsize(CHECKPOINT_PFAD)//1024//1024} MB)")


In [ ]:
#@title 3. Stimme hören { display-mode: "form" }
import re, numpy as np, soundfile as sf
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from IPython.display import Audio, display
from google.colab import files

LANGUAGE    = "tr"  #@param {type:"string"}
TEMPERATURE = 0.7   #@param {type:"slider", min:0.3, max:1.0, step:0.05}
REP_PENALTY = 5.0   #@param {type:"slider", min:1.0, max:10.0, step:0.5}

# ── Test-Text (mehrzeilig erlaubt) ──────────────────────────────
TEST_TEXT = """Şimdi bizde iki yön var. Ne denliyiz ve iki yönümüz var.
Bir değişmeyen kendileri her bir anda kendi olarak kalan.
Sabah kalktım, aynaya baktım. Dünkü ben ben miyim? Aynı ben mi?"""
# ────────────────────────────────────────────────────────────────

print("⬆️  Referenz-Clip hochladen (sauberer ~6–15s Schnipsel NUR deiner Stimme):")
up = files.upload()
SPEAKER_REF = [list(up.keys())[0]]

cfg = XttsConfig(); cfg.load_json(CONFIG)
xtts = Xtts.init_from_config(cfg)
# Fine-getunte Gewichte aus dem Trainer-Checkpoint laden
try:
    xtts.load_checkpoint(cfg, checkpoint_path=CHECKPOINT_PFAD, vocab_path=VOCAB, use_deepspeed=False)
except Exception as e:
    print("ℹ️  Direktes Laden fehlgeschlagen, extrahiere model-State ...", str(e)[:120])
    import torch
    ck = torch.load(CHECKPOINT_PFAD, map_location="cpu")
    state = ck.get("model", ck)
    tmp = "/content/_model_only.pth"; torch.save(state, tmp)
    xtts.load_checkpoint(cfg, checkpoint_path=tmp, vocab_path=VOCAB, use_deepspeed=False)
xtts.cuda()

gpt_latent, spk = xtts.get_conditioning_latents(audio_path=SPEAKER_REF)

def saetze(t, m=220):
    out=[]
    for s in re.split(r"(?<=[.!?;…])\s+", t.replace("\n"," ").strip()):
        s=s.strip()
        while len(s)>m:
            c=s.rfind(" ",0,m); c=c if c>0 else m; out.append(s[:c].strip()); s=s[c:].strip()
        if s: out.append(s)
    return out

pause=np.zeros(int(24000*0.3),dtype=np.float32); chunks=[]
S=saetze(TEST_TEXT); print(f"📝 {len(S)} Sätze")
for i,s in enumerate(S,1):
    r=xtts.inference(text=s, language=LANGUAGE, gpt_cond_latent=gpt_latent,
                     speaker_embedding=spk, temperature=TEMPERATURE, repetition_penalty=REP_PENALTY)
    chunks.append(np.asarray(r["wav"],dtype=np.float32)); chunks.append(pause)
    print(f"  ✓ {i}/{len(S)}")
audio=np.concatenate(chunks); sf.write("/content/test.wav",audio,24000)
print(f"🔊 {len(audio)/24000:.1f}s"); display(Audio("/content/test.wav",rate=24000))
